# 📦 E-Commerce EDA — Olist Brazilian Dataset
**Goal:** Understand data shape, nulls, distributions, and date ranges before building the pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

In [ ]:
from src.ingest import load_all
dfs = load_all()
print('Loaded tables:', list(dfs.keys()))

## 1. Shape & Dtypes

In [ ]:
for name, df in dfs.items():
    print(f"\n{'='*40}")
    print(f"{name}: {df.shape}")
    print(df.dtypes)
    print('\nTop 5 null columns:')
    print(df.isnull().mean().sort_values(ascending=False).head(5))

## 2. Orders — Date Range & Status Breakdown

In [ ]:
orders = dfs['orders']
print('Date range:')
print('  Min:', orders['order_purchase_timestamp'].min())
print('  Max:', orders['order_purchase_timestamp'].max())

print('\nOrder status breakdown:')
print(orders['order_status'].value_counts(normalize=True).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
orders['order_status'].value_counts().plot(kind='bar', ax=ax, color='#6366f1')
ax.set_title('Order Status Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 3. Monthly Order Volume

In [ ]:
orders['month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)
monthly = orders.groupby('month')['order_id'].count()

fig, ax = plt.subplots(figsize=(12, 4))
monthly.plot(ax=ax, marker='o', color='#6366f1')
ax.set_title('Monthly Order Volume', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Order Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Item Prices — Distribution & Outliers

In [ ]:
items = dfs['items']
print('Price stats:')
print(items['price'].describe())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
items['price'].clip(upper=500).hist(bins=50, ax=ax1, color='#10b981')
ax1.set_title('Price Distribution (clipped at R$500)')

items.boxplot(column='price', ax=ax2)
ax2.set_title('Price Boxplot')
plt.tight_layout()
plt.show()

## 5. Review Score Distribution

In [ ]:
reviews = dfs['reviews']
print('Review score distribution:')
print(reviews['review_score'].value_counts(normalize=True).sort_index().round(3))

fig, ax = plt.subplots(figsize=(7, 4))
reviews['review_score'].value_counts().sort_index().plot(kind='bar', ax=ax, color='#f59e0b')
ax.set_title('Review Score Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Score')
plt.tight_layout()
plt.show()

## 6. Key EDA Findings

*(Fill in your findings after running the notebook)*

- **Date range:** Sep 2016 – Oct 2018
- **Order status:** ~97% delivered, ~1% shipped, rest cancelled/unavailable
- **Price:** Highly right-skewed — median ~R$74, mean ~R$120, max ~R$6,735
- **Review score:** Positively skewed — ~57% give a 5-star rating
- **Nulls:** `order_delivered_customer_date` has nulls for non-delivered orders — expected
- **Growth:** Clear month-on-month growth in 2017, slight dip in late 2018